# Resolve a storage code explicitly

The database column is called money_unit; the domain field is called currency.

Run in a fresh kernel using the repository's Python environment. The matching script is `04_storage_mapping.py`.


## Prepare the model

These declarations supply the complete model needed by this case. Run this cell before the experiment.


In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")


## Run the experiment

The database column is called money_unit; the domain field is called currency.


In [2]:
from aoa.action_machine.domain import build

row = {"tag_key": "tag-1", "price": 9.99, "money_unit": "eur"}
value = CurrencyEnumEntity.by_code(row["money_unit"])
tag = build(
    row,
    PriceTagCoreEntity,
    lambda fields, data: {
        fields.id: data["tag_key"],
        fields.amount: data["price"],
        fields.currency: value,
    },
)
print(tag.id, tag.currency.code)
print("Shared value:", tag.currency is CurrencyEnumEntity.EUR)


tag-1 eur
Shared value: True


## Explain the result

The resource owns the mapping from storage columns to model fields. by_code raises KeyError for an unknown code; it never invents a default currency. Writing back uses tag.currency.code. No database connection is needed for this mapping experiment.
